# Modern NLP Pipeline for Training Programme Applications

This Google Colab notebook analyzes free-text motivation responses from applicants to a training programme. It uses semantic embeddings, BERTopic, KeyBERT, UMAP, HDBSCAN, and a transparent custom motivation scoring engine.

The notebook is designed to run top to bottom after uploading an Excel file. The only variable most users should change is `TEXT_COLUMN` in Section 3.

The goal is not sentiment analysis. The goal is to understand applicant motivations, major themes, applicant similarity, motivation quality, common keywords, and applicant clusters.

## Section 1: Installation

This section installs the modern NLP stack required for the analysis, downloads the spaCy English model, and downloads NLTK resources. In Google Colab, installation usually takes a few minutes.

In [1]:
# Install required libraries for Google Colab.
# Restart the runtime only if Colab asks you to after installation.
%pip install -q pandas numpy matplotlib plotly spacy nltk sentence-transformers bertopic umap-learn hdbscan keybert scikit-learn wordcloud openpyxl kaleido tqdm

# Download the spaCy English model used for lemmatization.
!python -m spacy download en_core_web_sm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 45.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
# Download NLTK resources used for stopwords and token handling.
import nltk

for resource_name in ["stopwords", "punkt", "punkt_tab"]:
    try:
        nltk.download(resource_name, quiet=True)
    except Exception as exc:
        print(f"Could not download NLTK resource {resource_name}: {exc}")

## Section 2: Imports

Imports are grouped by purpose to make the notebook easier to maintain. The notebook avoids TF-IDF, NMF, TextBlob, RAKE, and TextRank.

In [3]:
# Standard library imports
import os
import re
import math
import string
import warnings
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

# Data handling
import numpy as np
import pandas as pd

# Display helpers
from IPython.display import display, HTML
from google.colab import files
from tqdm.auto import tqdm

# Visualization
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from wordcloud import WordCloud

# NLP
import nltk
import spacy
from spacy.language import Language
from nltk.corpus import stopwords
from sentence_transformers import SentenceTransformer
from keybert import KeyBERT
from bertopic import BERTopic

# Machine learning and clustering
import umap
import hdbscan
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 220)

tqdm.pandas()

In [4]:
# Global visual style for publication-quality figures.
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update(
    {
        "figure.figsize": (12, 7),
        "figure.dpi": 140,
        "savefig.dpi": 300,
        "axes.titlesize": 16,
        "axes.labelsize": 12,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "legend.fontsize": 10,
        "font.family": "DejaVu Sans",
    }
)

PLOTLY_TEMPLATE = "plotly_white"
RANDOM_STATE = 42

## Section 3: Load Data

Upload an Excel workbook, read the first sheet by default, inspect the data, and automatically detect likely free-text columns. Change only `TEXT_COLUMN` if your form uses a different motivation question.

In [5]:
# User configuration: this should be the only variable most users need to change.
TEXT_COLUMN = "Why are you interested in participating in this training program?"
OUTPUT_EXCEL_PATH = "applications_analysis.xlsx"
EMBEDDINGS_PATH = "application_embeddings.npy"

# Optional: set to a specific sheet name or index. None uses the first sheet.
SHEET_NAME = 0

In [6]:
def upload_excel_file() -> str:
    """Upload one Excel file in Google Colab and return its local filename."""
    uploaded_files = files.upload()
    if not uploaded_files:
        raise FileNotFoundError("No file was uploaded. Please upload an Excel file.")

    excel_candidates = [
        filename for filename in uploaded_files.keys()
        if filename.lower().endswith((".xlsx", ".xls"))
    ]
    if not excel_candidates:
        raise ValueError("Please upload a .xlsx or .xls Excel file.")

    selected_file = excel_candidates[0]
    print(f"Loaded upload: {selected_file}")
    return selected_file


def read_excel_data(file_path: str, sheet_name: Any = 0) -> pd.DataFrame:
    """Read an Excel workbook into a dataframe with defensive error handling."""
    try:
        dataframe = pd.read_excel(file_path, sheet_name=sheet_name)
    except Exception as exc:
        raise RuntimeError(f"Could not read Excel file {file_path}: {exc}") from exc

    if dataframe.empty:
        raise ValueError("The uploaded Excel sheet is empty.")

    dataframe = dataframe.copy()
    dataframe.columns = [str(column).strip() for column in dataframe.columns]
    return dataframe


def display_dataframe_overview(dataframe: pd.DataFrame) -> None:
    """Display practical diagnostics for a newly loaded dataframe."""
    print(f"Shape: {dataframe.shape[0]:,} rows x {dataframe.shape[1]:,} columns")
    display(dataframe.head())

    print("\nDataframe info:")
    dataframe.info()

    missing_values = (
        dataframe.isna()
        .sum()
        .sort_values(ascending=False)
        .reset_index()
        .rename(columns={"index": "Column", 0: "Missing Values"})
    )
    missing_values["Missing Percent"] = (
        missing_values["Missing Values"] / len(dataframe) * 100
    ).round(2)
    display(missing_values)

In [7]:
def detect_text_columns(
    dataframe: pd.DataFrame,
    min_average_words: float = 4.0,
    min_non_null_ratio: float = 0.10,
) -> pd.DataFrame:
    """Detect likely free-text columns using dtype, coverage, and average word count."""
    candidates = []

    for column in dataframe.columns:
        series = dataframe[column].dropna().astype(str).str.strip()
        non_null_ratio = len(series) / max(len(dataframe), 1)
        if series.empty or non_null_ratio < min_non_null_ratio:
            continue

        word_counts = series.str.split().str.len()
        average_words = float(word_counts.mean()) if not word_counts.empty else 0.0
        median_words = float(word_counts.median()) if not word_counts.empty else 0.0
        unique_ratio = series.nunique() / max(len(series), 1)
        average_length = float(series.str.len().mean())

        is_likely_text = (
            dataframe[column].dtype == "object"
            and average_words >= min_average_words
            and average_length >= 20
            and unique_ratio >= 0.20
        )

        if is_likely_text:
            candidates.append(
                {
                    "Column": column,
                    "Non-Null Ratio": round(non_null_ratio, 3),
                    "Average Words": round(average_words, 2),
                    "Median Words": round(median_words, 2),
                    "Unique Ratio": round(unique_ratio, 3),
                    "Average Characters": round(average_length, 2),
                }
            )

    return pd.DataFrame(candidates).sort_values(
        by=["Average Words", "Unique Ratio"], ascending=False
    ) if candidates else pd.DataFrame()


def validate_text_column(dataframe: pd.DataFrame, text_column: str) -> str:
    """Validate the configured text column and provide helpful guidance if missing."""
    if text_column in dataframe.columns:
        return text_column

    detected_columns = detect_text_columns(dataframe)
    available_columns = "\n".join(f"- {column}" for column in dataframe.columns)
    detected_message = ""
    if not detected_columns.empty:
        detected_message = "\n\nLikely text columns detected:\n" + "\n".join(
            f"- {column}" for column in detected_columns["Column"].tolist()
        )

    raise ValueError(
        f"TEXT_COLUMN was not found: {text_column}\n\n"
        f"Available columns:\n{available_columns}{detected_message}"
    )

In [8]:
excel_file_path = upload_excel_file()
applications_df = read_excel_data(excel_file_path, sheet_name=SHEET_NAME)
display_dataframe_overview(applications_df)

text_column_report = detect_text_columns(applications_df)
print("Detected likely text columns:")
display(text_column_report)

TEXT_COLUMN = validate_text_column(applications_df, TEXT_COLUMN)
print(f"Using text column: {TEXT_COLUMN}")

Saving 2026_BRICS_ASTRONOMY_DATA_SCIENCE_AI_TRAINING_Application_form_1784104415.xlsx to 2026_BRICS_ASTRONOMY_DATA_SCIENCE_AI_TRAINING_Application_form_1784104415.xlsx
Loaded upload: 2026_BRICS_ASTRONOMY_DATA_SCIENCE_AI_TRAINING_Application_form_1784104415.xlsx
Shape: 1,220 rows x 9 columns


,Organization/Affiliation,Country of Residence,Highest Level of Education,"What is your profession, field of study, or area of work?",Why are you interested in participating in this training program?,This program requires every participant to have a working personal computer for hands-on practice. Do you have a personal computer?,"Do you have any experience with programming languages (e.g., Python)?","If yes, please list them and describe your level of expertise.",How does this program align with your long-term career goals or academic aspirations?
0,Ignatius Global School,Indonesia,Bachelor's Degree,Teaching Mathematics and Astronomy in Middle High School,Want to change job that align with astronomy major in science bachelor degree,Yes,Yes,"i has Certificate of Data analysis from SANBERCODE and achieve best trainee at that time, if i need to level my python, it in early expert because im still self learning in machine learning","It may make me open an opportunity to get study at master or PHD program after training, and hope can get stable research career for at least 20 or 30 years"
1,Surabaya Astronomy Club,Indonesia,Master's Degree,"Amateur Astronomer, Human Resource Development, Public Outreach & Science Communication",I want to study about data science and machine learning (ML) with practical applications to Astronomy,Yes,No,NaN,"In the future, I hope I can write a paper or do a research in the field of Astronomy using data science and machine learning (ML). It would be very useful for the amateur astronomer like me."
2,Institute of technology Bandung,Indonesia,Master's Degree,Student,Background study in mathematics and astronomy,Yes,Yes,"I have study in basic programming with python, i use an intermediete python to make an artificial intelligence in image processing. My final riset about classification of quality rice paddies using neurofuzzy, but it...",I hope get an project science in this field. Because i interest with this topic. I open to work
3,UIN Sutalnah Nahrasiyah Lhokseumawe,Indonesia,Bachelor's Degree,Student,Mempunyai ketertarikan mendalam tentang ML,Yes,Yes,NaN,Saya ingin membangun sebuah penelitian tesis machine learning dengan astronomi
4,Imah Noong Observatory,Indonesia,Bachelor's Degree,Amateur Astronomers,"I want to get more experience and knowledge, also the network across countries",Yes,Yes,Beginner,This program helps me to develop any ML-empowered astronomical software which useful to my work in amateur observatory



Dataframe info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1220 entries, 0 to 1219
Data columns (total 9 columns):
 #   Column                                                                                                                               Non-Null Count  Dtype 
---  ------                                                                                                                               --------------  ----- 
 0   Organization/Affiliation                                                                                                             1217 non-null   object
 1   Country of Residence                                                                                                                 1220 non-null   object
 2   Highest Level of Education                                                                                                           1220 non-null   object
 3   What is your profession, field of study, or area of work?          

,Column,Missing Values,Missing Percent
0,"If yes, please list them and describe your level of expertise.",296,24.26
1,Organization/Affiliation,3,0.25
2,"What is your profession, field of study, or area of work?",1,0.08
3,How does this program align with your long-term career goals or academic aspirations?,1,0.08
4,Country of Residence,0,0.00
5,Why are you interested in participating in this training program?,0,0.00
6,Highest Level of Education,0,0.00
7,"Do you have any experience with programming languages (e.g., Python)?",0,0.00
8,This program requires every participant to have a working personal computer for hands-on practice. Do you have a personal computer?,0,0.00


Detected likely text columns:


,Column,Non-Null Ratio,Average Words,Median Words,Unique Ratio,Average Characters
4,How does this program align with your long-term career goals or academic aspirations?,0.999,83.24,56.0,0.984,546.71
2,Why are you interested in participating in this training program?,1.000,52.08,28.0,0.972,338.00
3,"If yes, please list them and describe your level of expertise.",0.757,25.77,12.0,0.908,174.08
1,"What is your profession, field of study, or area of work?",0.999,8.51,3.0,0.759,61.31
0,Organization/Affiliation,0.998,4.01,3.0,0.718,30.49


Using text column: Why are you interested in participating in this training program?


## Section 4: Text Cleaning

This section creates one reusable cleaning function. It lowercases text, removes URLs, email addresses, punctuation, numbers, and extra whitespace, then applies spaCy lemmatization, stopword removal, and a minimum token-length filter.

In [9]:
# Load spaCy once. Disable parser and NER for faster lemmatization.
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

BASE_STOPWORDS = set(stopwords.words("english"))
CUSTOM_STOPWORDS = {
    "training", "programme", "program", "participating", "participate",
    "interested", "interest", "application", "applicant",
}
ALL_STOPWORDS = BASE_STOPWORDS.union(CUSTOM_STOPWORDS)

URL_PATTERN = re.compile(r"https?://\S+|www\.\S+", flags=re.IGNORECASE)
EMAIL_PATTERN = re.compile(r"\b[\w\.-]+@[\w\.-]+\.\w+\b", flags=re.IGNORECASE)
PUNCT_TRANSLATOR = str.maketrans("", "", string.punctuation)


def normalize_raw_text(value: Any) -> str:
    """Convert a cell value to a normalized raw string while preserving missing values."""
    if pd.isna(value):
        return ""
    return str(value).strip()


def clean_single_text(
    text: Any,
    nlp_model: Language,
    stopword_set: set,
    min_token_length: int = 3,
) -> str:
    """Clean and lemmatize one free-text response.

    Steps:
    1. Convert missing values to empty strings.
    2. Lowercase text.
    3. Remove URLs and email addresses.
    4. Remove punctuation and numbers.
    5. Lemmatize with spaCy.
    6. Remove stopwords, non-alpha tokens, and short tokens.
    """
    normalized_text = normalize_raw_text(text).lower()
    normalized_text = URL_PATTERN.sub(" ", normalized_text)
    normalized_text = EMAIL_PATTERN.sub(" ", normalized_text)
    normalized_text = normalized_text.translate(PUNCT_TRANSLATOR)
    normalized_text = re.sub(r"\d+", " ", normalized_text)
    normalized_text = re.sub(r"\s+", " ", normalized_text).strip()

    if not normalized_text:
        return ""

    document = nlp_model(normalized_text)
    tokens = []
    for token in document:
        lemma = token.lemma_.strip().lower()
        if (
            lemma
            and lemma.isalpha()
            and len(lemma) >= min_token_length
            and lemma not in stopword_set
            and not token.is_stop
        ):
            tokens.append(lemma)

    return " ".join(tokens)


def add_clean_text_features(
    dataframe: pd.DataFrame,
    text_column: str,
    nlp_model: Language,
    stopword_set: set,
) -> pd.DataFrame:
    """Add raw text, clean text, and word-count features to the dataframe."""
    result = dataframe.copy()
    result["raw_text"] = result[text_column].apply(normalize_raw_text)
    result["clean_text"] = result["raw_text"].progress_apply(
        lambda text: clean_single_text(text, nlp_model, stopword_set)
    )
    result["word_count"] = result["raw_text"].str.split().str.len().fillna(0).astype(int)
    result["clean_word_count"] = result["clean_text"].str.split().str.len().fillna(0).astype(int)
    return result

In [10]:
analysis_df = add_clean_text_features(applications_df, TEXT_COLUMN, nlp, ALL_STOPWORDS)

before_after_examples = analysis_df[[TEXT_COLUMN, "clean_text", "word_count", "clean_word_count"]].head(8)
display(before_after_examples)

initial_rows = len(analysis_df)
analysis_df = analysis_df[analysis_df["clean_text"].str.len() > 0].reset_index(drop=True)
removed_rows = initial_rows - len(analysis_df)
print(f"Removed {removed_rows:,} empty or unusable responses.")
print(f"Remaining responses: {len(analysis_df):,}")

  0%|          | 0/1220 [00:00<?, ?it/s]

,Why are you interested in participating in this training program?,clean_text,word_count,clean_word_count
0,Want to change job that align with astronomy major in science bachelor degree,want change job align astronomy major science bachelor degree,13,9
1,I want to study about data science and machine learning (ML) with practical applications to Astronomy,want study data science machine learn practical astronomy,16,8
2,Background study in mathematics and astronomy,background study mathematic astronomy,6,4
3,Mempunyai ketertarikan mendalam tentang ML,mempunyai ketertarikan mendalam tentang,5,4
4,"I want to get more experience and knowledge, also the network across countries",want experience knowledge network country,13,5
5,I am highly interested in participating in this training program because it offers a valuable opportunity to strengthen my skills in data science and machine learning applications in astronomy. Although my academic b...,highly offer valuable opportunity strengthen skill data science machine learning astronomy academic background pure physics currently work astronomical research environment recognize expertise astronomical datum proc...,101,50
6,"I am interested in joining this program because modern space science increasingly relies on large-scale data analysis and machine learning techniques. As a data scientist at Research Center for Space, BRIN, Indonesia...",join modern space science increasingly rely largescale datum analysis machine learn technique data scientist research center space brin indonesia regularly work complex observational model dataset strengthen skill da...,132,75
7,"As a Data Operational professional and Bachelor of Marine Science , I have spent my career ensuring data integrity and optimizing workflows. I am interested in this program because I want to bridge the gap between my...",data operational professional bachelor marine science spend career ensure datum integrity optimize workflow want bridge gap operational expertise complex world big datum astronomy,48,23


Removed 7 empty or unusable responses.
Remaining responses: 1,213


## Section 5: Semantic Embeddings

This section converts each cleaned response into a dense semantic vector using `all-MiniLM-L6-v2`. These embeddings power topic modeling, applicant similarity, and clustering.

In [11]:
def generate_sentence_embeddings(
    texts: Sequence[str],
    model_name: str = "all-MiniLM-L6-v2",
    batch_size: int = 64,
    output_path: Optional[str] = None,
) -> Tuple[np.ndarray, SentenceTransformer]:
    """Generate sentence-transformer embeddings and optionally save them to disk."""
    if not texts:
        raise ValueError("No texts were provided for embedding generation.")

    embedding_model = SentenceTransformer(model_name)
    embeddings = embedding_model.encode(
        list(texts),
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    if output_path:
        np.save(output_path, embeddings)
        print(f"Saved embeddings to {output_path}")

    return embeddings, embedding_model

In [12]:
texts_for_modeling = analysis_df["clean_text"].tolist()
embeddings, sentence_model = generate_sentence_embeddings(
    texts_for_modeling,
    model_name="all-MiniLM-L6-v2",
    batch_size=64,
    output_path=EMBEDDINGS_PATH,
)

print(f"Embedding matrix shape: {embeddings.shape}")
print(f"Embedding dimensions: {embeddings.shape[1]:,}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Saved embeddings to application_embeddings.npy
Embedding matrix shape: (1213, 384)
Embedding dimensions: 384


## Section 6: Theme Discovery with BERTopic

BERTopic discovers themes from semantic embeddings. UMAP reduces the embedding space, HDBSCAN finds dense topic regions, and BERTopic assigns each applicant a topic, topic name, and probability.

In [13]:
def choose_topic_model_parameters(number_of_documents: int) -> Dict[str, int]:
    """Choose robust UMAP/HDBSCAN parameters for small and large datasets."""
    if number_of_documents < 30:
        return {
            "n_neighbors": max(2, min(8, number_of_documents - 1)),
            "n_components": 3,
            "min_cluster_size": max(2, number_of_documents // 8),
            "min_samples": 1,
        }

    if number_of_documents < 200:
        return {
            "n_neighbors": 10,
            "n_components": 5,
            "min_cluster_size": max(5, number_of_documents // 20),
            "min_samples": 2,
        }

    return {
        "n_neighbors": 15,
        "n_components": 5,
        "min_cluster_size": max(10, number_of_documents // 50),
        "min_samples": 5,
    }


def fit_bertopic_model(
    texts: Sequence[str],
    embeddings_array: np.ndarray,
    embedding_model: SentenceTransformer,
) -> Tuple[BERTopic, List[int], np.ndarray]:
    """Fit BERTopic with explicit UMAP and HDBSCAN models."""
    params = choose_topic_model_parameters(len(texts))
    print(f"BERTopic parameters: {params}")

    umap_model = umap.UMAP(
        n_neighbors=params["n_neighbors"],
        n_components=params["n_components"],
        min_dist=0.0,
        metric="cosine",
        random_state=RANDOM_STATE,
    )
    hdbscan_model = hdbscan.HDBSCAN(
        min_cluster_size=params["min_cluster_size"],
        min_samples=params["min_samples"],
        metric="euclidean",
        cluster_selection_method="eom",
        prediction_data=True,
    )

    topic_model = BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        calculate_probabilities=True,
        verbose=True,
        nr_topics="auto",
    )

    topics, probabilities = topic_model.fit_transform(list(texts), embeddings_array)
    return topic_model, topics, probabilities


def extract_topic_probability(topic: int, probability_row: Any) -> float:
    """Extract the assigned topic probability from a BERTopic probability row."""
    if probability_row is None or topic == -1:
        return np.nan
    try:
        probability_array = np.asarray(probability_row)
        if probability_array.ndim == 0 or probability_array.size == 0:
            return np.nan
        return float(np.nanmax(probability_array))
    except Exception:
        return np.nan


def add_topic_assignments(
    dataframe: pd.DataFrame,
    topic_model: BERTopic,
    topics: Sequence[int],
    probabilities: Any,
) -> pd.DataFrame:
    """Add BERTopic IDs, names, probabilities, and representative-topic flags."""
    result = dataframe.copy()
    topic_info = topic_model.get_topic_info().copy()
    topic_name_lookup = dict(zip(topic_info["Topic"], topic_info["Name"]))

    result["topic_id"] = list(topics)
    result["topic_name"] = result["topic_id"].map(topic_name_lookup).fillna("Outlier / No Topic")

    if probabilities is not None:
        result["topic_probability"] = [
            extract_topic_probability(topic, probability_row)
            for topic, probability_row in zip(topics, probabilities)
        ]
    else:
        result["topic_probability"] = np.nan

    return result

In [14]:
topic_model, topics, topic_probabilities = fit_bertopic_model(
    texts_for_modeling,
    embeddings,
    sentence_model,
)

analysis_df = add_topic_assignments(analysis_df, topic_model, topics, topic_probabilities)

topic_info_df = topic_model.get_topic_info()
display(topic_info_df.head(20))
display(analysis_df[["topic_id", "topic_name", "topic_probability", "raw_text"]].head(10))

2026-07-15 13:50:36,865 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


BERTopic parameters: {'n_neighbors': 15, 'n_components': 5, 'min_cluster_size': 24, 'min_samples': 5}


2026-07-15 13:50:48,712 - BERTopic - Dimensionality - Completed ✓
2026-07-15 13:50:48,713 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-07-15 13:50:48,774 - BERTopic - Cluster - Completed ✓
2026-07-15 13:50:48,775 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-07-15 13:50:48,815 - BERTopic - Representation - Completed ✓
2026-07-15 13:50:48,816 - BERTopic - Topic reduction - Reducing number of topics
2026-07-15 13:50:48,826 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-07-15 13:50:48,866 - BERTopic - Representation - Completed ✓
2026-07-15 13:50:48,869 - BERTopic - Topic reduction - Reduced number of topics from 10 to 10


,Topic,Count,Name,Representation,Representative_Docs
0,-1,160,-1_skill_want_astronomy_learn,"[skill, want, astronomy, learn, help, right, childhood, cod, datum, language]","[intreste astronomy right childhood learn python language right childhood want learn thing relate astronomy, intreste astronomy right childhood learn python language right childhood want learn thing relate astronomy,..."
1,0,681,0_datum_astronomy_research_learn,"[datum, astronomy, research, learn, science, machine, astronomical, skill, dataset, learning]",[modern astronomy increasingly rely large complex dataset make datum science machine learn essential tool astronomical research develop strong computational skill area python programming statistical analysis machine ...
2,1,68,1_knowledge_research_improve_work,"[knowledge, research, improve, work, experience, gain, enhance, people, new, skill]","[gain knowledge research work, knowledge, knowledge]"
3,2,63,2_data_learning_machine_skill,"[data, learning, machine, skill, analytic, science, knowledge, want, enhance, like]","[data science, enhance data science skill, enhance skill knowledge data science machine learning]"
4,3,59,3_datum_analysis_learn_science,"[datum, analysis, learn, science, learning, machine, analytic, visualization, want, process]","[learn datum analysis, learn datum analysis machine learning, python datum science datum analysis machine learning]"
5,4,44,4_skill_learn_new_improve,"[skill, learn, new, improve, want, thing, like, upgrade, sharpen, increase]","[learn improve skill, want learn new skill, want learn new skill]"
6,5,41,5_career_skill_gain_help,"[career, skill, gain, help, development, industry, knowledge, personal, growth, develop]","[opportunity gain important skill essential career growth, student eager learn new skill broaden knowledge alot career field take learn new help future career skill come handy line relevant current world, develop ski..."
7,6,38,6_datum_skill_analysis_enhance,"[datum, skill, analysis, enhance, rightful, gain, data, analytic, science, improve]",[data analysis want develop strong skill work data datum analysis important skill today technologydriven world learn tool like datum visualization datum cleaning statistical analysis help understand extract meaningfu...
8,7,35,7_python_want_skill_programming,"[python, want, skill, programming, learning, work, learn, analytic, data, machine]","[want learn programming python, learn python, want learn python]"
9,8,24,8_astronomy_hobby_cosmology_universe,"[astronomy, hobby, cosmology, universe, poststudie, astrostatistic, incline, dara, project, society]","[astronomy, astronomy, astronomy]"


,topic_id,topic_name,topic_probability,raw_text
0,0,0_datum_astronomy_research_learn,1.000000,Want to change job that align with astronomy major in science bachelor degree
1,0,0_datum_astronomy_research_learn,0.357817,I want to study about data science and machine learning (ML) with practical applications to Astronomy
2,8,8_astronomy_hobby_cosmology_universe,1.000000,Background study in mathematics and astronomy
3,-1,-1_skill_want_astronomy_learn,NaN,Mempunyai ketertarikan mendalam tentang ML
4,1,1_knowledge_research_improve_work,0.526457,"I want to get more experience and knowledge, also the network across countries"
5,0,0_datum_astronomy_research_learn,0.678719,I am highly interested in participating in this training program because it offers a valuable opportunity to strengthen my skills in data science and machine learning applications in astronomy. Although my academic b...
6,0,0_datum_astronomy_research_learn,1.000000,"I am interested in joining this program because modern space science increasingly relies on large-scale data analysis and machine learning techniques. As a data scientist at Research Center for Space, BRIN, Indonesia..."
7,-1,-1_skill_want_astronomy_learn,NaN,"As a Data Operational professional and Bachelor of Marine Science , I have spent my career ensuring data integrity and optimizing workflows. I am interested in this program because I want to bridge the gap between my..."
8,0,0_datum_astronomy_research_learn,0.336955,deepening knowledge of data science in astronomy
9,0,0_datum_astronomy_research_learn,1.000000,"I am interested in participating in this training program because it provides a unique opportunity to explore the intersection of astronomy, data science, and machine learning. I have a strong curiosity about the uni..."


In [15]:
def get_topic_keywords_dataframe(topic_model: BERTopic, top_n_words: int = 10) -> pd.DataFrame:
    """Return a tidy dataframe of BERTopic keywords by topic."""
    rows = []
    for topic_id in topic_model.get_topic_info()["Topic"].tolist():
        if topic_id == -1:
            continue
        for rank, keyword_score in enumerate(topic_model.get_topic(topic_id)[:top_n_words], start=1):
            keyword, score = keyword_score
            rows.append(
                {
                    "topic_id": topic_id,
                    "rank": rank,
                    "keyword": keyword,
                    "score": score,
                }
            )
    return pd.DataFrame(rows)


def display_representative_documents(topic_model: BERTopic, max_topics: int = 10) -> None:
    """Display representative documents for the most frequent topics."""
    topic_info = topic_model.get_topic_info()
    topic_ids = topic_info[topic_info["Topic"] != -1]["Topic"].head(max_topics).tolist()

    for topic_id in topic_ids:
        print(f"\nTopic {topic_id}: {topic_model.get_topic_info(topic_id).iloc[0]['Name']}")
        representative_docs = topic_model.get_representative_docs(topic_id)
        for index, document in enumerate(representative_docs[:3], start=1):
            print(f"  {index}. {document[:350]}...")

In [16]:
topic_keywords_df = get_topic_keywords_dataframe(topic_model, top_n_words=12)
display(topic_keywords_df.head(40))

display_representative_documents(topic_model, max_topics=8)

,topic_id,rank,keyword,score
0,0,1,datum,0.045151
1,0,2,astronomy,0.041206
2,0,3,research,0.037997
3,0,4,learn,0.035641
4,0,5,science,0.035402
5,0,6,machine,0.035010
6,0,7,astronomical,0.031220
7,0,8,skill,0.030052
8,0,9,dataset,0.029869
9,0,10,learning,0.029789



Topic 0: 0_datum_astronomy_research_learn
  1. modern astronomy increasingly rely large complex dataset make datum science machine learn essential tool astronomical research develop strong computational skill area python programming statistical analysis machine learning allow well analyse interpret observational datum background physics astronomy research experience observational astronomy astr...
  2. bric astronomy virtual offer unique opportunity integrate datum science machine learn realworld scientific postgraduate student pursue msc computer science specialization data analytic particularly apply analytical machine learn technique complex dataset astronomy generate vast amount datum allow explore modern datum science method extract meaningf...
  3. offer valuable opportunity strengthen skill data science machine learning apply realworld scientific datum health informatic student particularly data analytic datadriven method generate insight support research different scientific f

In [17]:
# Interactive BERTopic visualizations. These render directly in Colab.
try:
    topic_model.visualize_barchart(top_n_topics=12, n_words=8).show()
except Exception as exc:
    print(f"Could not render BERTopic barchart: {exc}")

try:
    topic_model.visualize_hierarchy().show()
except Exception as exc:
    print(f"Could not render BERTopic hierarchy: {exc}")

try:
    topic_model.visualize_heatmap().show()
except Exception as exc:
    print(f"Could not render BERTopic heatmap: {exc}")

try:
    topic_model.visualize_topics().show()
except Exception as exc:
    print(f"Could not render BERTopic topic map: {exc}")

## Section 7: Motivation Scoring

This section implements a transparent scoring engine from 0 to 100. It does not use sentiment analysis. The score rewards evidence of specific goals, career objectives, astronomy relevance, data science relevance, programming relevance, research relevance, future impact, clarity, and specificity.

In [18]:
@dataclass(frozen=True)
class ScoringCriterion:
    """Configuration for one keyword-backed scoring criterion."""
    name: str
    weight: float
    keywords: Tuple[str, ...]
    description: str


SCORING_CRITERIA = [
    ScoringCriterion(
        name="specific_goals",
        weight=10,
        keywords=("goal", "aim", "objective", "plan", "learn", "develop", "build", "improve", "skill"),
        description="Mentions concrete learning or development goals.",
    ),
    ScoringCriterion(
        name="career_objectives",
        weight=10,
        keywords=("career", "profession", "job", "work", "industry", "employ", "future", "path"),
        description="Connects the programme to career plans.",
    ),
    ScoringCriterion(
        name="astronomy_relevance",
        weight=9,
        keywords=("astronomy", "astronomer", "space", "telescope", "galaxy", "star", "cosmology", "astrophysics", "universe"),
        description="Connects motivation to astronomy or space science.",
    ),
    ScoringCriterion(
        name="data_science_relevance",
        weight=10,
        keywords=("data", "dataset", "analysis", "analytics", "machine", "learning", "model", "statistics", "visualization"),
        description="Shows interest in data science and analytical methods.",
    ),
    ScoringCriterion(
        name="programming_relevance",
        weight=7,
        keywords=("python", "programming", "coding", "code", "software", "algorithm", "computational", "automation"),
        description="Mentions programming or computational skills.",
    ),
    ScoringCriterion(
        name="research_relevance",
        weight=9,
        keywords=("research", "project", "study", "investigate", "evidence", "scientific", "experiment", "publication"),
        description="Connects programme participation to research practice.",
    ),
    ScoringCriterion(
        name="future_impact",
        weight=9,
        keywords=("impact", "community", "contribute", "solve", "benefit", "advance", "innovation", "society", "transform"),
        description="Explains intended future contribution or impact.",
    ),
]


def bounded_keyword_score(tokens: Sequence[str], keywords: Sequence[str], maximum_score: float) -> float:
    """Score a keyword criterion with saturation to avoid rewarding repetition."""
    token_counts = Counter(tokens)
    matched_count = sum(token_counts.get(keyword, 0) for keyword in keywords)
    unique_matches = sum(1 for keyword in keywords if keyword in token_counts)

    if matched_count == 0:
        return 0.0

    repetition_component = min(matched_count / 4, 1.0) * 0.45
    breadth_component = min(unique_matches / 3, 1.0) * 0.55
    return maximum_score * (repetition_component + breadth_component)


def score_response_length(word_count: int) -> float:
    """Reward responses that contain enough detail without requiring excessive length."""
    if word_count <= 0:
        return 0.0
    if word_count < 20:
        return 4.0 * (word_count / 20)
    if word_count <= 80:
        return 4.0 + 8.0 * ((word_count - 20) / 60)
    if word_count <= 180:
        return 12.0 + 4.0 * ((word_count - 80) / 100)
    return 16.0


def score_clarity(raw_text: str) -> float:
    """Estimate clarity from sentence length, alphabetic ratio, and readable structure."""
    normalized = normalize_raw_text(raw_text)
    if not normalized:
        return 0.0

    words = normalized.split()
    sentences = re.split(r"[.!?]+", normalized)
    sentences = [sentence.strip() for sentence in sentences if sentence.strip()]
    average_sentence_length = len(words) / max(len(sentences), 1)
    alphabetic_ratio = sum(character.isalpha() or character.isspace() for character in normalized) / max(len(normalized), 1)

    sentence_score = 1.0 if 8 <= average_sentence_length <= 30 else 0.65
    alphabetic_score = min(max((alphabetic_ratio - 0.55) / 0.40, 0), 1)
    structure_score = 1.0 if len(sentences) >= 1 else 0.5

    return 10.0 * ((0.45 * sentence_score) + (0.35 * alphabetic_score) + (0.20 * structure_score))


def score_specificity(raw_text: str, clean_text: str) -> float:
    """Reward concrete details such as named tools, domains, goals, and multiword specificity."""
    raw = normalize_raw_text(raw_text)
    clean_tokens = clean_text.split()
    if not clean_tokens:
        return 0.0

    named_detail_patterns = [
        r"\bpython\b", r"\bmachine learning\b", r"\bdata science\b", r"\bresearch project\b",
        r"\bastronomy\b", r"\bastrophysics\b", r"\btelescope\b", r"\bstatistics\b",
        r"\bvisuali[sz]ation\b", r"\bcareer\b", r"\bmasters?\b", r"\bphd\b",
    ]
    pattern_matches = sum(bool(re.search(pattern, raw, flags=re.IGNORECASE)) for pattern in named_detail_patterns)
    lexical_diversity = len(set(clean_tokens)) / max(len(clean_tokens), 1)

    pattern_score = min(pattern_matches / 5, 1.0) * 6.0
    diversity_score = min(lexical_diversity / 0.70, 1.0) * 4.0
    return pattern_score + diversity_score


def compute_motivation_score(raw_text: str, clean_text: str, word_count: int) -> Dict[str, float]:
    """Compute a 0-100 motivation score and criterion-level score components."""
    tokens = clean_text.split()
    component_scores = {
        "response_length_score": score_response_length(word_count),
        "clarity_score": score_clarity(raw_text),
        "specificity_score": score_specificity(raw_text, clean_text),
    }

    for criterion in SCORING_CRITERIA:
        component_scores[f"{criterion.name}_score"] = bounded_keyword_score(
            tokens=tokens,
            keywords=criterion.keywords,
            maximum_score=criterion.weight,
        )

    total_score = sum(component_scores.values())
    component_scores["motivation_score"] = round(float(min(max(total_score, 0), 100)), 2)
    return component_scores


def add_motivation_scores(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Add motivation score and all component scores to the analysis dataframe."""
    result = dataframe.copy()
    score_rows = [
        compute_motivation_score(row["raw_text"], row["clean_text"], int(row["word_count"]))
        for _, row in tqdm(result.iterrows(), total=len(result), desc="Scoring responses")
    ]
    score_df = pd.DataFrame(score_rows)
    return pd.concat([result.reset_index(drop=True), score_df.reset_index(drop=True)], axis=1)

In [19]:
analysis_df = add_motivation_scores(analysis_df)

display(
    analysis_df[
        ["raw_text", "word_count", "motivation_score", "clarity_score", "specificity_score"]
    ].sort_values("motivation_score", ascending=False).head(10)
)

display(pd.DataFrame([criterion.__dict__ for criterion in SCORING_CRITERIA]))

Scoring responses:   0%|          | 0/1213 [00:00<?, ?it/s]

,raw_text,word_count,motivation_score,clarity_score,specificity_score
43,"I am interested in participating in this training program because modern astronomy increasingly relies on large and complex datasets, making data science and machine learning essential tools for astronomical research...",223,91.38,10.000,9.925845
641,"I am currently pursuing a Master of Science degree in Physics with a research focus in High Energy Astrophysics, particularly the study of the prompt emission and afterglow of Gamma-Ray Bursts (GRBs) using observatio...",566,89.73,10.000,9.957597
747,"I am currently a first-year Integrated BS–MS student at IISER Thiruvananthapuram with an intended major in Physics and a strong interest in cosmology, astrophysics, and gravitational physics. I am particularly intere...",216,87.93,10.000,10.000000
289,"I am highly interested in the BRICS Astronomy program because, as a secondary school teacher of physics, I have witnessed many young Cameroonians struggle to secure good jobs due to a lack of integration of data scie...",290,87.49,10.000,8.800000
693,"My primary academic interest lies in physics, a passion that began with my early fascination with astronomy and the fundamental questions about the universe it raises. During my undergraduate studies in physics, I de...",228,87.36,10.000,6.400000
137,"My primary interest lies in astronomy and astrophysics, particularly in understanding complex physical phenomena through computational and data-driven approaches. Modern astronomical research increasingly relies on t...",214,85.85,10.000,8.800000
791,Astronomy has always been an special interest in my life. From when i used to look up to the stars as a grounding technique until i started asking questions about constellations not even knowing what they are. Partic...,1405,84.03,10.000,5.822708
1132,I am highly interested in participating in this training programme because it combines two fields I am passionate about—astronomy and data science. As a physics student with a growing interest in areas like astrobiol...,145,83.42,10.000,8.800000
238,"I am interested in participating in this training program because it offers a unique opportunity to explore the intersection of data science, machine learning, and astronomy, fields that are increasingly shaping the ...",227,82.62,10.000,7.600000
115,My interest in this program comes from my growing involvement in astrophysical data analysis and my desire to develop stronger computational skills for astronomy research. As an undergraduate student in Information T...,181,82.22,8.425,8.800000


,name,weight,keywords,description
0,specific_goals,10,"(goal, aim, objective, plan, learn, develop, build, improve, skill)",Mentions concrete learning or development goals.
1,career_objectives,10,"(career, profession, job, work, industry, employ, future, path)",Connects the programme to career plans.
2,astronomy_relevance,9,"(astronomy, astronomer, space, telescope, galaxy, star, cosmology, astrophysics, universe)",Connects motivation to astronomy or space science.
3,data_science_relevance,10,"(data, dataset, analysis, analytics, machine, learning, model, statistics, visualization)",Shows interest in data science and analytical methods.
4,programming_relevance,7,"(python, programming, coding, code, software, algorithm, computational, automation)",Mentions programming or computational skills.
5,research_relevance,9,"(research, project, study, investigate, evidence, scientific, experiment, publication)",Connects programme participation to research practice.
6,future_impact,9,"(impact, community, contribute, solve, benefit, advance, innovation, society, transform)",Explains intended future contribution or impact.


## Section 8: Motivation Classification

Applicants are categorized from the numeric score into four review-friendly groups: Highly Motivated, Moderately Motivated, Needs Review, and Weak Motivation.

In [20]:
def classify_motivation(score: float) -> str:
    """Convert a numeric motivation score into an interpretable category."""
    if pd.isna(score):
        return "Needs Review"
    if score >= 75:
        return "Highly Motivated"
    if score >= 55:
        return "Moderately Motivated"
    if score >= 35:
        return "Needs Review"
    return "Weak Motivation"


analysis_df["motivation_category"] = analysis_df["motivation_score"].apply(classify_motivation)

category_summary = (
    analysis_df["motivation_category"]
    .value_counts()
    .rename_axis("motivation_category")
    .reset_index(name="count")
)
category_summary["percent"] = (category_summary["count"] / len(analysis_df) * 100).round(2)
display(category_summary)

,motivation_category,count,percent
0,Weak Motivation,692,57.05
1,Needs Review,279,23.00
2,Moderately Motivated,203,16.74
3,Highly Motivated,39,3.22


## Section 9: Keyword Extraction with KeyBERT

KeyBERT extracts meaningful phrases from each applicant response and from the full corpus. This section also creates a keyword frequency dataframe, a bar chart, and a word cloud.

In [21]:
def build_keybert_model(sentence_embedding_model: SentenceTransformer) -> KeyBERT:
    """Create a KeyBERT model backed by the same sentence-transformer model."""
    return KeyBERT(model=sentence_embedding_model)


def extract_keywords_for_document(
    keyword_model: KeyBERT,
    document: str,
    top_n: int = 6,
    keyphrase_ngram_range: Tuple[int, int] = (1, 3),
) -> List[str]:
    """Extract top keywords from one document with graceful fallback for short text."""
    if not document or len(document.split()) < 3:
        return []

    try:
        keywords = keyword_model.extract_keywords(
            document,
            keyphrase_ngram_range=keyphrase_ngram_range,
            stop_words="english",
            top_n=top_n,
            use_mmr=True,
            diversity=0.65,
        )
        return [keyword for keyword, score in keywords]
    except Exception:
        return []


def add_keybert_keywords(
    dataframe: pd.DataFrame,
    keyword_model: KeyBERT,
    text_column: str = "clean_text",
    top_n: int = 6,
) -> pd.DataFrame:
    """Add semicolon-separated KeyBERT keywords for each applicant."""
    result = dataframe.copy()
    keyword_lists = []
    for document in tqdm(result[text_column].tolist(), desc="Extracting applicant keywords"):
        keyword_lists.append(extract_keywords_for_document(keyword_model, document, top_n=top_n))

    result["keywords"] = ["; ".join(keywords) for keywords in keyword_lists]
    result["keyword_list"] = keyword_lists
    return result


def build_keyword_frequency_dataframe(keyword_lists: Sequence[Sequence[str]]) -> pd.DataFrame:
    """Create a frequency dataframe from applicant-level keyword lists."""
    counter = Counter()
    for keywords in keyword_lists:
        counter.update(keywords)

    frequency_df = pd.DataFrame(counter.items(), columns=["keyword", "frequency"])
    if frequency_df.empty:
        return frequency_df
    return frequency_df.sort_values("frequency", ascending=False).reset_index(drop=True)


def extract_corpus_keywords(
    keyword_model: KeyBERT,
    documents: Sequence[str],
    top_n: int = 30,
) -> pd.DataFrame:
    """Extract top keywords across the entire application corpus."""
    corpus = ". ".join(document for document in documents if document)
    keywords = keyword_model.extract_keywords(
        corpus,
        keyphrase_ngram_range=(1, 3),
        stop_words="english",
        top_n=top_n,
        use_mmr=True,
        diversity=0.70,
    )
    return pd.DataFrame(keywords, columns=["keyword", "score"])

In [ ]:
keyword_model = build_keybert_model(sentence_model)
analysis_df = add_keybert_keywords(analysis_df, keyword_model, text_column="clean_text", top_n=6)

keyword_frequency_df = build_keyword_frequency_dataframe(analysis_df["keyword_list"])
corpus_keywords_df = extract_corpus_keywords(keyword_model, analysis_df["clean_text"].tolist(), top_n=30)

display(analysis_df[["raw_text", "keywords"]].head(10))
display(keyword_frequency_df.head(30))
display(corpus_keywords_df.head(30))

Extracting applicant keywords:   0%|          | 0/1213 [00:00<?, ?it/s]

In [ ]:
def plot_keyword_frequency(keyword_frequency: pd.DataFrame, top_n: int = 25) -> None:
    """Plot the most frequent extracted keywords."""
    if keyword_frequency.empty:
        print("No keywords available to plot.")
        return

    plot_df = keyword_frequency.head(top_n).sort_values("frequency", ascending=True)
    fig = px.bar(
        plot_df,
        x="frequency",
        y="keyword",
        orientation="h",
        template=PLOTLY_TEMPLATE,
        title=f"Top {top_n} Extracted Keywords",
        labels={"frequency": "Frequency", "keyword": "Keyword"},
    )
    fig.update_layout(height=700, margin=dict(l=180, r=40, t=70, b=40))
    fig.show()


def plot_word_cloud(keyword_frequency: pd.DataFrame) -> None:
    """Create a word cloud from keyword frequencies."""
    if keyword_frequency.empty:
        print("No keywords available for word cloud.")
        return

    frequency_dict = dict(zip(keyword_frequency["keyword"], keyword_frequency["frequency"]))
    word_cloud = WordCloud(
        width=1400,
        height=800,
        background_color="white",
        colormap="viridis",
        max_words=150,
        prefer_horizontal=0.90,
    ).generate_from_frequencies(frequency_dict)

    plt.figure(figsize=(14, 8))
    plt.imshow(word_cloud, interpolation="bilinear")
    plt.axis("off")
    plt.title("Keyword Word Cloud", pad=20)
    plt.show()

In [ ]:
plot_keyword_frequency(keyword_frequency_df, top_n=25)
plot_word_cloud(keyword_frequency_df)

## Section 10: Applicant Clustering

This section clusters applicants using semantic embeddings. HDBSCAN is used by default because it can discover irregular clusters and mark outliers. KMeans fallback is available if HDBSCAN finds too few clusters.

In [ ]:
def reduce_embeddings_to_2d(embeddings_array: np.ndarray) -> np.ndarray:
    """Reduce semantic embeddings to two dimensions for plotting."""
    if len(embeddings_array) < 4:
        return embeddings_array[:, :2]

    reducer = umap.UMAP(
        n_neighbors=max(2, min(15, len(embeddings_array) - 1)),
        n_components=2,
        min_dist=0.08,
        metric="cosine",
        random_state=RANDOM_STATE,
    )
    return reducer.fit_transform(embeddings_array)


def cluster_with_hdbscan(embeddings_array: np.ndarray) -> np.ndarray:
    """Cluster embeddings with HDBSCAN using dataset-aware parameters."""
    number_of_documents = len(embeddings_array)
    min_cluster_size = max(5, number_of_documents // 40) if number_of_documents >= 80 else max(2, number_of_documents // 10)
    min_samples = 2 if number_of_documents < 200 else 5

    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        metric="euclidean",
        cluster_selection_method="eom",
        prediction_data=True,
    )
    return clusterer.fit_predict(embeddings_array)


def choose_kmeans_cluster_count(embeddings_array: np.ndarray, max_clusters: int = 8) -> int:
    """Choose a reasonable KMeans cluster count using silhouette score."""
    number_of_documents = len(embeddings_array)
    if number_of_documents < 6:
        return 2

    upper_bound = min(max_clusters, number_of_documents - 1)
    candidate_scores = []
    for cluster_count in range(2, upper_bound + 1):
        labels = KMeans(
            n_clusters=cluster_count,
            random_state=RANDOM_STATE,
            n_init="auto",
        ).fit_predict(embeddings_array)
        try:
            score = silhouette_score(embeddings_array, labels, metric="cosine")
        except Exception:
            score = -1
        candidate_scores.append((cluster_count, score))

    return max(candidate_scores, key=lambda item: item[1])[0]


def cluster_applicants(embeddings_array: np.ndarray) -> Tuple[np.ndarray, str]:
    """Cluster applicants with HDBSCAN, falling back to KMeans when needed."""
    if len(embeddings_array) < 4:
        return np.zeros(len(embeddings_array), dtype=int), "single_cluster"

    hdbscan_labels = cluster_with_hdbscan(embeddings_array)
    discovered_clusters = sorted(label for label in set(hdbscan_labels) if label != -1)
    non_outlier_ratio = np.mean(hdbscan_labels != -1)

    if len(discovered_clusters) >= 2 and non_outlier_ratio >= 0.35:
        return hdbscan_labels, "hdbscan"

    cluster_count = choose_kmeans_cluster_count(embeddings_array)
    kmeans_labels = KMeans(
        n_clusters=cluster_count,
        random_state=RANDOM_STATE,
        n_init="auto",
    ).fit_predict(embeddings_array)
    return kmeans_labels, "kmeans_fallback"


def summarize_clusters(dataframe: pd.DataFrame, text_column: str = "raw_text") -> pd.DataFrame:
    """Create cluster sizes, average scores, top keywords, and representative applicants."""
    summary_rows = []
    for cluster_id, group in dataframe.groupby("cluster"):
        keyword_counter = Counter()
        for keywords in group["keyword_list"]:
            keyword_counter.update(keywords)

        representative = (
            group.sort_values("motivation_score", ascending=False)[text_column]
            .head(3)
            .tolist()
        )
        summary_rows.append(
            {
                "cluster": cluster_id,
                "size": len(group),
                "average_motivation_score": round(group["motivation_score"].mean(), 2),
                "top_keywords": "; ".join(keyword for keyword, count in keyword_counter.most_common(8)),
                "representative_applicants": " | ".join(text[:250] for text in representative),
            }
        )

    return pd.DataFrame(summary_rows).sort_values("size", ascending=False)

In [ ]:
embedding_2d = reduce_embeddings_to_2d(embeddings)
cluster_labels, clustering_method = cluster_applicants(embeddings)

analysis_df["umap_x"] = embedding_2d[:, 0]
analysis_df["umap_y"] = embedding_2d[:, 1]
analysis_df["cluster"] = cluster_labels.astype(str)

cluster_summary_df = summarize_clusters(analysis_df)
print(f"Clustering method used: {clustering_method}")
display(cluster_summary_df)

In [ ]:
def plot_cluster_scatter(dataframe: pd.DataFrame) -> None:
    """Create an interactive 2D applicant similarity map."""
    hover_columns = ["topic_name", "motivation_score", "motivation_category", "keywords"]
    fig = px.scatter(
        dataframe,
        x="umap_x",
        y="umap_y",
        color="cluster",
        symbol="motivation_category",
        hover_data=hover_columns,
        template=PLOTLY_TEMPLATE,
        title="Applicant Similarity Map Based on Semantic Embeddings",
        labels={"umap_x": "UMAP 1", "umap_y": "UMAP 2", "cluster": "Cluster"},
        opacity=0.82,
    )
    fig.update_traces(marker=dict(size=9, line=dict(width=0.5, color="white")))
    fig.update_layout(height=750, legend_title_text="Cluster")
    fig.show()


plot_cluster_scatter(analysis_df)

## Section 11: Visualizations

This section creates publication-quality figures for topic frequencies, motivation scores, keyword frequencies, clusters, and optional demographic heatmaps when relevant columns exist.

In [ ]:
def plot_topic_frequencies(dataframe: pd.DataFrame, top_n: int = 15) -> None:
    """Plot the most frequent BERTopic themes."""
    topic_counts = (
        dataframe.groupby(["topic_id", "topic_name"])
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
        .head(top_n)
    )
    topic_counts["topic_label"] = topic_counts["topic_id"].astype(str) + ": " + topic_counts["topic_name"]
    topic_counts = topic_counts.sort_values("count", ascending=True)

    fig = px.bar(
        topic_counts,
        x="count",
        y="topic_label",
        orientation="h",
        template=PLOTLY_TEMPLATE,
        title=f"Top {top_n} Topic Frequencies",
        labels={"count": "Applicants", "topic_label": "Topic"},
    )
    fig.update_layout(height=720, margin=dict(l=220, r=40, t=70, b=40))
    fig.show()


def plot_motivation_distribution(dataframe: pd.DataFrame) -> None:
    """Plot motivation score distribution by motivation category."""
    fig = px.histogram(
        dataframe,
        x="motivation_score",
        color="motivation_category",
        nbins=24,
        marginal="box",
        template=PLOTLY_TEMPLATE,
        title="Motivation Score Distribution",
        labels={"motivation_score": "Motivation Score", "motivation_category": "Category"},
    )
    fig.update_layout(height=650, bargap=0.05)
    fig.show()


def find_column_by_keywords(dataframe: pd.DataFrame, keywords: Sequence[str]) -> Optional[str]:
    """Find the first column whose name contains one of the supplied keywords."""
    normalized_columns = {column: column.lower().strip() for column in dataframe.columns}
    for column, normalized in normalized_columns.items():
        if any(keyword.lower() in normalized for keyword in keywords):
            return column
    return None


def plot_optional_topic_heatmap(
    dataframe: pd.DataFrame,
    column_keywords: Sequence[str],
    title_prefix: str,
    max_categories: int = 15,
) -> None:
    """Plot a category-vs-topic heatmap if a matching metadata column exists."""
    metadata_column = find_column_by_keywords(dataframe, column_keywords)
    if metadata_column is None:
        print(f"No {title_prefix.lower()} column detected. Skipping heatmap.")
        return

    working_df = dataframe[[metadata_column, "topic_name"]].dropna().copy()
    working_df[metadata_column] = working_df[metadata_column].astype(str).str.strip()
    working_df = working_df[working_df[metadata_column] != ""]
    if working_df.empty:
        print(f"Detected {metadata_column}, but it has no usable values.")
        return

    top_categories = working_df[metadata_column].value_counts().head(max_categories).index
    working_df = working_df[working_df[metadata_column].isin(top_categories)]

    heatmap_df = pd.crosstab(working_df[metadata_column], working_df["topic_name"])
    fig = px.imshow(
        heatmap_df,
        text_auto=True,
        aspect="auto",
        template=PLOTLY_TEMPLATE,
        title=f"{title_prefix} vs Topic Heatmap",
        labels=dict(x="Topic", y=metadata_column, color="Applicants"),
    )
    fig.update_layout(height=max(500, 35 * len(heatmap_df)), margin=dict(l=160, r=40, t=80, b=160))
    fig.show()

In [ ]:
plot_topic_frequencies(analysis_df, top_n=15)
plot_motivation_distribution(analysis_df)
plot_keyword_frequency(keyword_frequency_df, top_n=25)
plot_cluster_scatter(analysis_df)

plot_optional_topic_heatmap(
    analysis_df,
    column_keywords=["country", "nationality", "location"],
    title_prefix="Country",
)

plot_optional_topic_heatmap(
    analysis_df,
    column_keywords=["education", "degree", "qualification", "study", "university"],
    title_prefix="Education",
)

## Section 12: Export Results

This section exports a single Excel workbook containing the enriched applicant-level results and supporting summary tables.

In [ ]:
def prepare_export_dataframe(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Prepare applicant-level results for Excel export."""
    result = dataframe.copy()
    if "keyword_list" in result.columns:
        result = result.drop(columns=["keyword_list"])

    preferred_columns = [
        "raw_text",
        "clean_text",
        "topic_id",
        "topic_name",
        "topic_probability",
        "motivation_score",
        "motivation_category",
        "keywords",
        "cluster",
        "word_count",
        "clean_word_count",
        "umap_x",
        "umap_y",
    ]
    component_columns = [
        column for column in result.columns
        if column.endswith("_score") and column != "motivation_score"
    ]
    original_columns = [
        column for column in result.columns
        if column not in preferred_columns and column not in component_columns
    ]
    ordered_columns = original_columns + preferred_columns + component_columns
    ordered_columns = list(dict.fromkeys(column for column in ordered_columns if column in result.columns))
    return result[ordered_columns]


def export_analysis_results(
    dataframe: pd.DataFrame,
    output_path: str,
    topic_info: pd.DataFrame,
    topic_keywords: pd.DataFrame,
    keyword_frequency: pd.DataFrame,
    corpus_keywords: pd.DataFrame,
    cluster_summary: pd.DataFrame,
    category_summary_table: pd.DataFrame,
) -> None:
    """Export applicant-level and summary results to one Excel workbook."""
    export_df = prepare_export_dataframe(dataframe)

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        export_df.to_excel(writer, sheet_name="Applicant Results", index=False)
        topic_info.to_excel(writer, sheet_name="Topic Summary", index=False)
        topic_keywords.to_excel(writer, sheet_name="Topic Keywords", index=False)
        keyword_frequency.to_excel(writer, sheet_name="Keyword Frequency", index=False)
        corpus_keywords.to_excel(writer, sheet_name="Corpus Keywords", index=False)
        cluster_summary.to_excel(writer, sheet_name="Cluster Summary", index=False)
        category_summary_table.to_excel(writer, sheet_name="Motivation Categories", index=False)

    print(f"Exported analysis workbook: {output_path}")


def download_file_if_colab(path: str) -> None:
    """Download a file from Colab when available."""
    try:
        files.download(path)
    except Exception as exc:
        print(f"File was created at {path}, but automatic download was not available: {exc}")

In [ ]:
export_analysis_results(
    dataframe=analysis_df,
    output_path=OUTPUT_EXCEL_PATH,
    topic_info=topic_info_df,
    topic_keywords=topic_keywords_df,
    keyword_frequency=keyword_frequency_df,
    corpus_keywords=corpus_keywords_df,
    cluster_summary=cluster_summary_df,
    category_summary_table=category_summary,
)

download_file_if_colab(OUTPUT_EXCEL_PATH)

## Final Notes

The exported workbook contains applicant-level results, topic summaries, keyword summaries, cluster summaries, and motivation categories. The scoring engine is transparent by design: edit `SCORING_CRITERIA`, `score_response_length`, `score_clarity`, or `score_specificity` to align the scoring rubric with a specific selection policy.